# UK-Air CSV → PowerBI Transform

Transforms UK-Air hourly measurement CSVs into a PowerBI-friendly star schema.

| Output file | Description |
|---|---|
| `location.csv` | One row per monitoring site — dimension table |
| `measurement_type.csv` | One row per distinct measurement type — dimension table |
| `data.csv` | One row per measurement per site per hour — fact table |

**Status codes:** `V` = Verified · `P` = Provisionally Verified · `N` = Not Verified · `S` = Suspect

## 1 — Configuration

Set input/output paths and which years to process.

In [15]:
import csv
import re
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_DIR  = Path(".")   # folder containing the yearly CSVs
OUTPUT_DIR = Path(".")   # where location.csv and data.csv will be written

# ── Years to process ───────────────────────────────────────────────────────
YEARS = [2020, 2021, 2022, 2023, 2024]

# ── Raw file structure ─────────────────────────────────────────────────────
# Row indices (0-based) in the 10-row metadata header
META_ROWS = {
    "site_name":       3,
    "latitude":        4,
    "longitude":       5,
    "site_type":       6,
    "zone":            7,
    "agglomeration":   8,
    "local_authority": 9,
}
COLUMN_HEADER_ROW = 10   # 0-based index of the measurement-type header row

print("Configuration loaded.")

Configuration loaded.


## 2 — Helper functions

In [16]:
def strip_unit(value: str) -> str:
    """Remove unit suffix from a measurement value, keeping only the numeric part.
    
    Example: '44.9 V ugm-3 (TEOM FDMS)' → '44.9'
    """
    if not value or value.strip().lower() in ("", "no data"):
        return ""
    match = re.match(r"^\s*([0-9]+\.?[0-9]*(?:[eE][+-]?[0-9]+)?)", value.strip())
    return match.group(1) if match else ""


def parse_metadata(rows: list) -> dict:
    """Extract site metadata from the header rows.
    
    Returns a dict mapping column_index → metadata dict for every
    column that has a non-empty site name.
    """
    sites = {}
    site_name_row = rows[META_ROWS["site_name"]]
    for col_idx, cell in enumerate(site_name_row):
        name = cell.strip()
        if name and name.lower() != "site name":
            meta = {"site_name": name}
            for field, row_idx in META_ROWS.items():
                if field == "site_name":
                    continue
                val = rows[row_idx][col_idx] if col_idx < len(rows[row_idx]) else ""
                meta[field] = val.strip()
            sites[col_idx] = meta
    return sites


def build_column_map(header_row: list, site_col_indices: list) -> list:
    """Map each site column to its measurement type and paired status column.
    
    Data columns come in pairs: (value, status).
    Returns a list of dicts with keys: site_col, measurement_type, status_col.
    """
    mapping = []
    for col_idx in site_col_indices:
        mtype = header_row[col_idx].strip() if col_idx < len(header_row) else "Unknown"
        status_col = col_idx + 1 if (col_idx + 1) < len(header_row) else None
        mapping.append({
            "site_col": col_idx,
            "measurement_type": mtype,
            "status_col": status_col,
        })
    return mapping


print("Helper functions defined.")

Helper functions defined.


## 3 — Parse all yearly files

Reads each CSV, extracts site metadata, and collects all measurement rows.

In [17]:
all_locations        = {}   # site_name → metadata dict (deduped across years)
location_id_map      = {}   # site_name → integer location_id
next_location_id     = 1
measurement_type_map = {}   # measurement_type string → integer measurement_type_id
next_mtype_id        = 1
all_data_rows        = []

for year in YEARS:
    fpath = INPUT_DIR / f"{year}.csv"
    if not fpath.exists():
        print(f"  [SKIP] {fpath} not found")
        continue

    print(f"Processing {year}.csv ...")

    with open(fpath, newline="", encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))

    # Parse site metadata from the header block
    sites_meta       = parse_metadata(rows)
    site_col_indices = sorted(sites_meta.keys())

    # Register any new locations (sites can appear in multiple years)
    for col_idx in site_col_indices:
        name = sites_meta[col_idx]["site_name"]
        if name not in location_id_map:
            location_id_map[name] = next_location_id
            all_locations[name]   = sites_meta[col_idx]
            next_location_id += 1

    # Build the column → measurement-type map for this year's layout
    col_header = rows[COLUMN_HEADER_ROW]
    col_map    = build_column_map(col_header, site_col_indices)

    # Iterate over data rows and flatten into one row per site per hour
    data_rows = rows[COLUMN_HEADER_ROW + 1:]
    for data_row in data_rows:
        if not data_row or not data_row[0].strip():
            continue
        date = data_row[0].strip()
        time = data_row[1].strip() if len(data_row) > 1 else ""

        for i, col_idx in enumerate(site_col_indices):
            site_name   = sites_meta[col_idx]["site_name"]
            location_id = location_id_map[site_name]
            mtype       = col_map[i]["measurement_type"]
            status_col  = col_map[i]["status_col"]

            # Register new measurement types on first encounter
            if mtype not in measurement_type_map:
                measurement_type_map[mtype] = next_mtype_id
                next_mtype_id += 1  # noqa: F841 (updated via enclosing scope)

            raw_value  = data_row[col_idx].strip() if col_idx < len(data_row) else ""
            raw_status = data_row[status_col].strip() if (status_col and status_col < len(data_row)) else ""

            value  = strip_unit(raw_value)
            status = raw_status[0] if raw_status else ""   # keep only V/P/N/S

            all_data_rows.append({
                "location_id":          location_id,
                "measurement_type_id":  measurement_type_map[mtype],
                "year":                 year,
                "date":                 date,
                "time":                 time,
                "value":                value,
                "status":               status,
            })

    print(f"  → {len(site_col_indices)} sites · {len(data_rows):,} time rows")

print(f"\nTotal unique sites            : {len(all_locations)}")
print(f"Total unique measurement types: {len(measurement_type_map)}")
print(f"Total measurement rows collected: {len(all_data_rows):,}")

Processing 2020.csv ...
  → 10 sites · 8,788 time rows
Processing 2021.csv ...
  → 10 sites · 8,764 time rows
Processing 2022.csv ...
  → 10 sites · 8,764 time rows
Processing 2023.csv ...
  → 10 sites · 8,764 time rows
Processing 2024.csv ...
  → 9 sites · 8,788 time rows

Total unique sites            : 10
Total unique measurement types: 3
Total measurement rows collected: 429,745


## 4 — Preview: location table

In [18]:
import pandas as pd

loc_records = []
for name, meta in all_locations.items():
    row = {"location_id": location_id_map[name]}
    row.update(meta)
    loc_records.append(row)

df_locations = pd.DataFrame(loc_records, columns=[
    "location_id", "site_name", "latitude", "longitude",
    "site_type", "zone", "agglomeration", "local_authority"
])

df_locations

,location_id,site_name,latitude,longitude,site_type,zone,agglomeration,local_authority
0,1,London Bexley,51.466030,0.184806,Suburban Background,Greater London,Greater London Urban Area,Bexley
1,2,London Bloomsbury,51.522290,-0.125889,Urban Background,Greater London,Greater London Urban Area,Camden
2,3,London Eltham,51.452580,0.070766,Suburban Background,Greater London,Greater London Urban Area,Greenwich
3,4,London Harlington,51.488790,-0.441614,Urban Industrial,Greater London,Greater London Urban Area,Hillingdon
4,5,London Hillingdon,51.496330,-0.460861,Urban Background,Greater London,Greater London Urban Area,Hillingdon
5,6,London Honor Oak Park,51.449674,-0.037418,Urban Background,Greater London,Greater London Urban Area,Lewisham
6,7,London Marylebone Road,51.522530,-0.154611,Urban Traffic,Greater London,Greater London Urban Area,Westminster
7,8,London N. Kensington,51.521050,-0.213419,Urban Background,Greater London,Greater London Urban Area,Kensington
8,9,London Teddington Bushy Park,51.425286,-0.345606,Urban Background,Greater London,Greater London Urban Area,Richmond
9,10,London Westminster,51.494670,-0.131931,Urban Background,Greater London,Greater London Urban Area,Westminster


## 5 — Preview: measurement_type table

In [19]:
# Parse the full measurement name into a short label and unit
# e.g. 'PM2.5 particulate matter (Hourly measured)' → label='PM2.5', unit='µg/m³'
UNIT_MAP = {
    "pm10":           "µg/m³",
    "pm2.5":          "µg/m³",
    "carbon monoxide": "mg/m³",
}

def derive_label(name: str) -> str:
    """Return a short display label from the full measurement type string."""
    name_lower = name.lower()
    if "pm2.5" in name_lower:
        return "PM2.5"
    if "pm10" in name_lower:
        return "PM10"
    if "carbon monoxide" in name_lower:
        return "CO"
    return name  # fallback: keep original

def derive_unit(name: str) -> str:
    """Return the measurement unit from the full measurement type string."""
    name_lower = name.lower()
    for keyword, unit in UNIT_MAP.items():
        if keyword in name_lower:
            return unit
    return ""

mtype_records = [
    {
        "measurement_type_id": mid,
        "measurement_type":    mtype,
        "label":               derive_label(mtype),
        "unit":                derive_unit(mtype),
    }
    for mtype, mid in sorted(measurement_type_map.items(), key=lambda x: x[1])
]

df_measurement_types = pd.DataFrame(mtype_records, columns=[
    "measurement_type_id", "measurement_type", "label", "unit"
])

df_measurement_types

,measurement_type_id,measurement_type,label,unit
0,1,PM10 particulate matter (Hourly measured),PM10,µg/m³
1,2,Carbon monoxide,CO,mg/m³
2,3,PM2.5 particulate matter (Hourly measured),PM2.5,µg/m³


## 6 — Preview: data table (first 20 rows)

In [20]:
df_data = pd.DataFrame(all_data_rows, columns=[
    "location_id", "measurement_type_id", "year", "date", "time", "value", "status"
])

print(f"Shape: {df_data.shape[0]:,} rows × {df_data.shape[1]} columns")
print(f"\nStatus breakdown:")
print(df_data["status"].value_counts().to_string())

df_data.head(20)

Shape: 429,745 rows × 7 columns

Status breakdown:
status
V    387129
      42616


,location_id,measurement_type_id,year,date,time,value,status
0,1,1,2020,2020-01-01,01:00:00,,
1,2,1,2020,2020-01-01,01:00:00,44.9,V
2,3,1,2020,2020-01-01,01:00:00,,
3,4,1,2020,2020-01-01,01:00:00,66.335,V
4,5,1,2020,2020-01-01,01:00:00,,
5,6,1,2020,2020-01-01,01:00:00,68.3,V
6,7,2,2020,2020-01-01,01:00:00,,V
7,8,2,2020,2020-01-01,01:00:00,0.213586,V
8,9,1,2020,2020-01-01,01:00:00,,V
9,10,3,2020,2020-01-01,01:00:00,42,V


## 7 — Write output files

In [21]:
# Write location.csv
loc_path = OUTPUT_DIR / "location.csv"
df_locations.to_csv(loc_path, index=False, encoding="utf-8")
print(f"✓ location.csv written          → {len(df_locations)} rows")

# Write measurement_type.csv
mtype_path = OUTPUT_DIR / "measurement_type.csv"
df_measurement_types.to_csv(mtype_path, index=False, encoding="utf-8")
print(f"✓ measurement_type.csv written  → {len(df_measurement_types)} rows")

# Write data.csv
data_path = OUTPUT_DIR / "data.csv"
df_data.to_csv(data_path, index=False, encoding="utf-8")
print(f"✓ data.csv written              → {len(df_data):,} rows")

print("\nDone. Ready to import into PowerBI.")

✓ location.csv written          → 10 rows
✓ measurement_type.csv written  → 3 rows
✓ data.csv written              → 429,745 rows

Done. Ready to import into PowerBI.
